# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json). It contains comprehensive, structured information about clinical and pathological variables in a cohort of 77 cancer survivors. All exploration references dataset entities via their `@id`.


In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant[tabular]>=1.0

## 1. Data Loading
Let's load the FAIR^2 dataset's metadata and available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Dataset instance

# Print overview from metadata (access properties directly)
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review the available record sets, their `@id`s, associated fields (columns), and the first few records of each. This helps us understand the dataset structure and available variables.

In [ ]:
# List all available record sets and their fields by @id
def list_record_sets(ds):
    print("Available Record Sets and Fields:")
    for rs in ds.record_sets:
        print(f"- RecordSet name: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Fields/columns (@id):")
        for fld in rs.fields:
            print(f"      - {fld.name} (@id: {fld.id}, dataType: {fld.data_type})")
        print("  Sample record:")
        try:
            sample = next(ds.records(record_set=rs.id))
            pprint(sample)
        except Exception as e:
            print("    (No records available or error loading)")
        print("")

list_record_sets(dataset)

## 3. Data Extraction
Extract the records for each record set into pandas `DataFrame`s for further analysis.

*Use the exact RecordSet and field `@id`s shown in the output above!*

In [ ]:
# List all record set @ids
record_sets = [rs.id for rs in dataset.record_sets]

# If there is only one principal record set (with actual data), use it; else, let user select
if not record_sets:
    raise ValueError("No record sets found in the schema.")
elif len(record_sets) == 1:
    main_record_set_id = record_sets[0]
else:
    # Try to infer main recordset by searching for the one with most fields or a key name
    _rs_by_fields = sorted(dataset.record_sets, key=lambda rs: len(rs.fields), reverse=True)
    main_record_set_id = _rs_by_fields[0].id
    print("Multiple record sets found, selecting the one with most fields:\n", main_record_set_id)

# Load all records into DataFrames by their @id
dataframes = {}
for rs_id in record_sets:
    rs_records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(rs_records)
    dataframes[rs_id] = df
    print(f"Loaded RecordSet {rs_id}: Shape {df.shape}")

# Show the columns/fields of the main record set and its head
print(f"\nColumns in main data table ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Now, let's process the clinical data. We'll:
* Select a numeric field (e.g., `Age` or a suitable one by @id).
* Filter records (e.g., keep cases where the numeric field is above a threshold).
* Normalize the field.
* Optionally, group by a categorical attribute such as `Sex` or `AnatomicalLocation` if available.

⚠️ All fields must be referenced by their `@id`. Replace `<numeric_field_id>` and `<group_field_id>` as appropriate using output from section 2.

In [ ]:
# Select a numeric field by its @id (e.g., 'age')
numeric_field_id = None
group_field_id = None
main_fields = {f.id:f for f in [rs for rs in dataset.record_sets if rs.id==main_record_set_id][0].fields}

# Try to detect an 'age' field or another integer/float
for fid, f in main_fields.items():
    if 'age' in fid.lower():
        numeric_field_id = fid
        break
if numeric_field_id is None:
    # Fallback: select the first Integer/Float field
    for fid, f in main_fields.items():
        if f.data_type.lower() in {"integer", "float", "number"}:
            numeric_field_id = fid
            break
if numeric_field_id is None:
    raise RuntimeError("Could not determine a numeric field in the main record set.")

# Try to detect a group-by field: 'sex', 'anatomical', or other category
for fid, f in main_fields.items():
    if any(x in fid.lower() for x in ['sex', 'anatomical', 'location']):
        group_field_id = fid
        break

df = dataframes[main_record_set_id]
df = df.copy()

# Filter records: numeric_field > threshold (auto threshold = 10 for age, else 0)
if 'age' in numeric_field_id.lower():
    threshold = 40
else:
    threshold = df[numeric_field_id].median() if df[numeric_field_id].dtype in [float, int] else 0

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized values for {numeric_field_id}:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by category if available
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(f"mean_{numeric_field_id}")
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and compare groups if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# If a group field was found, visualize comparisons
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we:
* Loaded the FAIR^2 dataset using `mlcroissant`
* Explored its metadata, record sets, and fields (by `@id`)
* Loaded records into pandas DataFrames for flexible analysis
* Performed basic EDA: filtered and normalized a main numeric field; grouped by category if available
* Visualized distributions and group comparisons

This demonstrates how the Croissant standard and the `mlcroissant` library enable robust, reproducible FAIR data science using explicit references to all dataset components.